# Resonate dataset preparation

Prepare a **user–item interaction dataset** from the Last.fm 360K plays data for training the Resonate (LightGCN) model.

**LightGCN** uses implicit feedback: each (user, item) pair is a positive interaction (edge in the user–item bipartite graph). This notebook:

1. Loads and cleans the plays data
2. Maps users and items to integer indices
3. Optionally filters by minimum interactions per user/item
4. Splits into train/test (no cold-start in test)
5. Saves the dataset for training (edge lists + metadata)

## 1. Setup and load raw plays data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATASET_DIR = Path("../dataset")
PLAYS_FILE = DATASET_DIR / "usersha1-artmbid-artname-plays.tsv"
OUT_DIR = Path("../data")
OUT_DIR.mkdir(exist_ok=True)

# Sample size (use None to load full file; 500k is a good balance for experimentation)
NROWS = None

plays = pd.read_csv(
    PLAYS_FILE,
    sep="\t",
    nrows=NROWS,
    header=None,
    names=["user_sha1", "artist_mbid", "artist_name", "plays"],
)
print("Raw shape:", plays.shape)
plays.head()

Raw shape: (17535655, 4)


,user_sha1,artist_mbid,artist_name,plays
0,00000c289a1829a808ac09c00daf10bc3c4e223b,3bd73256-3905-4f3a-97e2-8b341527f805,betty blowtorch,2137
1,00000c289a1829a808ac09c00daf10bc3c4e223b,f2fb0ff0-5679-42ec-a55c-15109ce6e320,die Ärzte,1099
2,00000c289a1829a808ac09c00daf10bc3c4e223b,b3ae82c2-e60b-4551-a76d-6620f1b456aa,melissa etheridge,897
3,00000c289a1829a808ac09c00daf10bc3c4e223b,3d6bbeb7-f90e-4d10-b440-e153c0d10b53,elvenking,717
4,00000c289a1829a808ac09c00daf10bc3c4e223b,bbd2ffd7-17f4-4506-8572-c1ea58c3f9a8,juliette & the licks,706


## 2. Clean and binarize interactions

- Drop rows with missing or empty `artist_mbid`.
- For LightGCN we use **implicit feedback**: each (user, artist) row is one positive interaction. We keep unique (user, item) pairs (one edge per user–artist, ignoring play count for the graph).

In [2]:
# Drop missing artist id
plays = plays.dropna(subset=["artist_mbid"])
plays = plays[plays["artist_mbid"].astype(str).str.len() > 0]

# One row per (user, artist) with summed play count (so DB can show correct artist/user plays)
edges = plays.groupby(["user_sha1", "artist_mbid"], as_index=False).agg(plays=("plays", "sum"))
edges = edges.reset_index(drop=True)
print("Unique (user, item) interactions:", len(edges))
edges.head()

Unique (user, item) interactions: 17309151


,user_sha1,artist_mbid,plays
0,00000c289a1829a808ac09c00daf10bc3c4e223b,0639533a-0402-40ba-b6e0-18b067198b73,403
1,00000c289a1829a808ac09c00daf10bc3c4e223b,0fb62639-4143-443b-8779-6867a1d08230,183
2,00000c289a1829a808ac09c00daf10bc3c4e223b,144ef525-85e9-40c3-8335-02c32d0861f3,182
3,00000c289a1829a808ac09c00daf10bc3c4e223b,21f3573f-10cf-44b3-aeaa-26cccd8448b5,507
4,00000c289a1829a808ac09c00daf10bc3c4e223b,295a3ae3-9e81-4cff-a36f-8d48b8fb4dcf,288


## 3. Map to integer indices

Map each `user_sha1` and `artist_mbid` to contiguous integers `0 .. n_users-1` and `0 .. n_items-1` (required for LightGCN).

In [4]:
user_cats = edges["user_sha1"].astype("category")
item_cats = edges["artist_mbid"].astype("category")
edges["user_idx"] = user_cats.cat.codes.values
edges["item_idx"] = item_cats.cat.codes.values

n_users = int(edges["user_idx"].max()) + 1
n_items = int(edges["item_idx"].max()) + 1
print(f"n_users = {n_users}, n_items = {n_items}, n_edges = {len(edges)}")
edges.head()

n_users = 358858, n_items = 160112, n_edges = 17309151


,user_sha1,artist_mbid,plays,user_idx,item_idx
0,00000c289a1829a808ac09c00daf10bc3c4e223b,0639533a-0402-40ba-b6e0-18b067198b73,403,0,3818
1,00000c289a1829a808ac09c00daf10bc3c4e223b,0fb62639-4143-443b-8779-6867a1d08230,183,0,9810
2,00000c289a1829a808ac09c00daf10bc3c4e223b,144ef525-85e9-40c3-8335-02c32d0861f3,182,0,12602
3,00000c289a1829a808ac09c00daf10bc3c4e223b,21f3573f-10cf-44b3-aeaa-26cccd8448b5,507,0,21202
4,00000c289a1829a808ac09c00daf10bc3c4e223b,295a3ae3-9e81-4cff-a36f-8d48b8fb4dcf,288,0,25904


## 4. Optional: filter by minimum interactions

Keep only users with ≥ `min_user_inter` and items with ≥ `min_item_inter` interactions to reduce noise and cold-start. Re-index so indices remain contiguous.

In [5]:
MIN_USER_INTER = 5   # keep users with at least this many interactions
MIN_ITEM_INTER = 5   # keep items (artists) with at least this many interactions

user_counts = edges["user_idx"].value_counts()
item_counts = edges["item_idx"].value_counts()
keep_users = user_counts[user_counts >= MIN_USER_INTER].index.values
keep_items = item_counts[item_counts >= MIN_ITEM_INTER].index.values

edges = edges[
    edges["user_idx"].isin(keep_users) & edges["item_idx"].isin(keep_items)
].copy()

# Re-map to contiguous indices 0 .. n_users-1, 0 .. n_items-1
old_user_idx = edges["user_idx"].values
old_item_idx = edges["item_idx"].values
_, edges["user_idx"] = np.unique(old_user_idx, return_inverse=True)
_, edges["item_idx"] = np.unique(old_item_idx, return_inverse=True)

n_users = int(edges["user_idx"].max()) + 1
n_items = int(edges["item_idx"].max()) + 1
print(f"After filter: n_users = {n_users}, n_items = {n_items}, n_edges = {len(edges)}")

After filter: n_users = 358619, n_items = 87660, n_edges = 17173166


## 5. Train / test split

Random 80/20 split. Restrict test set to (user, item) pairs where both user and item appear in the training set so we can evaluate without cold-start.

In [6]:
SEED = 42
np.random.seed(SEED)

idx = np.random.permutation(len(edges))
split = int(0.8 * len(edges))
train_idx, test_idx = idx[:split], idx[split:]

train_edges = edges.iloc[train_idx][["user_idx", "item_idx", "plays"]].copy()
test_edges = edges.iloc[test_idx][["user_idx", "item_idx", "plays"]].copy()

train_users = set(train_edges["user_idx"])
train_items = set(train_edges["item_idx"])
test_edges = test_edges[
    test_edges["user_idx"].isin(train_users) & test_edges["item_idx"].isin(train_items)
]

print(f"Train edges: {len(train_edges)}, Test edges: {len(test_edges)}")

Train edges: 13738532, Test edges: 3434624


## 6. Save dataset for training

Outputs are written under the **`rec_`** prefix (recommendation) so filenames stay project-agnostic. Save train/test edge lists as NumPy (`.npz`) for PyTorch and CSV for inspection.

In [7]:
# NumPy arrays for model training (user indices, item indices)
train_user = train_edges["user_idx"].values.astype(np.int64)
train_item = train_edges["item_idx"].values.astype(np.int64)
test_user = test_edges["user_idx"].values.astype(np.int64)
test_item = test_edges["item_idx"].values.astype(np.int64)

out_npz = OUT_DIR / "rec_dataset.npz"
np.savez(
    out_npz,
    train_user=train_user,
    train_item=train_item,
    test_user=test_user,
    test_item=test_item,
    n_users=np.int64(n_users),
    n_items=np.int64(n_items),
)
print(f"Saved: {out_npz}")
print("Keys:", list(np.load(out_npz, allow_pickle=False).keys()))

Saved: ../data/rec_dataset.npz
Keys: ['train_user', 'train_item', 'test_user', 'test_item', 'n_users', 'n_items']


In [8]:
# Optional: save train/test as CSV for inspection
train_edges.to_csv(OUT_DIR / "rec_train_edges.csv", index=False)
test_edges.to_csv(OUT_DIR / "rec_test_edges.csv", index=False)

# Summary
summary = {
    "n_users": n_users,
    "n_items": n_items,
    "n_train_edges": len(train_edges),
    "n_test_edges": len(test_edges),
}
pd.DataFrame([summary]).to_csv(OUT_DIR / "rec_dataset_summary.csv", index=False)
print("Summary:", summary)

Summary: {'n_users': 358619, 'n_items': 87660, 'n_train_edges': 13738532, 'n_test_edges': 3434624}


## 7. Load the dataset (example)

Use this snippet in your training script or another notebook to load the prepared data.

In [9]:
# Example: load for LightGCN training
data = np.load(OUT_DIR / "rec_dataset.npz", allow_pickle=False)
train_user = data["train_user"]
train_item = data["train_item"]
test_user = data["test_user"]
test_item = data["test_item"]
n_users = int(data["n_users"])
n_items = int(data["n_items"])

# Build adjacency or edge index for your framework, e.g. PyTorch:
# edge_index = np.stack([train_user, train_item + n_users], axis=0)  # bipartite: items offset by n_users
print("Loaded. Example: first 5 train edges (user, item):")
print(np.stack([train_user[:5], train_item[:5]], axis=1))

Loaded. Example: first 5 train edges (user, item):
[[331818  80414]
 [ 50596  51275]
 [246609  43116]
 [ 34761  73635]
 [214694  21672]]
